In [3]:
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import (
    Ridge, Lasso, ElasticNet
)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_log_error, make_scorer
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

def rmsle(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))
rmsle_scorer = make_scorer(rmsle, greater_is_better=False)

import sys
import os
sys.path.append(os.path.abspath('..'))
import importlib
import preprocessing
importlib.reload(preprocessing)
from preprocessing import prepare_features
X, y, X_test, test_ids = prepare_features()

warnings.filterwarnings('ignore')

cv = KFold(n_splits=5, shuffle=True, random_state=42)
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True))
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])
def create_pipe(model, preprocessor, scale=False):
    steps = [('preprocessor', preprocessor)]

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)

def create_model(model, preprocessor, scale=False):
    return TransformedTargetRegressor(
        regressor=create_pipe(model, preprocessor, scale=scale),
        func=np.log1p,
        inverse_func=np.expm1
    )

best_estimators = {
    'LightGBM': create_model(
        LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=3,
            num_leaves=7,
            random_state=42,
            verbose=-1,
            n_jobs=1
        ),
        preprocessor,
        scale=False
    ),

    'CatBoost': create_model(
        CatBoostRegressor(
            iterations=500,
            depth=6,
            learning_rate=0.1,
            verbose=0,
            random_state=42
        ),
        preprocessor,
        scale=False
    ),

    'ElasticNet': create_model(
        ElasticNet(
            alpha=0.01,
            l1_ratio=0.5,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    ),

    'XGBoost': create_model(
        XGBRegressor(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=3,
            colsample_bytree=0.8,
            subsample=0.8,
            random_state=42,
            objective='reg:squarederror',
            eval_metric='rmse',
            n_jobs=1
        ),
        preprocessor,
        scale=False
    ),

    'Lasso': create_model(
        Lasso(
            alpha=0.01,
            random_state=42,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    ),

    'Ridge': create_model(
        Ridge(
            alpha=100,
            random_state=42,
            max_iter=20000
        ),
        preprocessor,
        scale=True
    )
}

top_models = [
    ('lgbm', best_estimators['LightGBM']),
    ('catboost', best_estimators['CatBoost']),
    ('elasticnet', best_estimators['ElasticNet']),
    ('xgb', best_estimators['XGBoost']),
    ('lasso', best_estimators['Lasso']),
    ('ridge', best_estimators['Ridge']),
]

ensemble_results = []
def ensemble_res(name, model):
    scores = cross_val_score(model, X, y, cv = cv, scoring = rmsle_scorer, n_jobs = 1)
    mean_score = -scores.mean()
    std_score = scores.std()
    
    ensemble_results.append({
        'Ensemble': name,
        'RMSLE': mean_score,
        'Std': std_score
    })
    print(f'[{name}] RMSLE: {mean_score:.4f} (+/- {std_score:.4f})')

In [4]:
#Voting_regressor
from sklearn.ensemble import VotingRegressor

voting = VotingRegressor(estimators = top_models)
ensemble_res('Voting Regressor', voting)

[Voting Regressor] RMSLE: 0.1089 (+/- 0.0080)


In [5]:
#Weighted_averaging
weighted_boosters = VotingRegressor(
    estimators = [
        ('catboost', best_estimators['CatBoost']),
        ('lgbm', best_estimators['LightGBM']),
        ('xgb', best_estimators['XGBoost'])
    ],
    weights = [3, 2, 2]
)

ensemble_res('Weighted averaging Top Boosters', weighted_boosters)

[Weighted averaging Top Boosters] RMSLE: 0.1128 (+/- 0.0085)


In [ ]:
#Stacking_regressor
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

stacking_ridge = StackingRegressor(
    estimators = top_models,
    final_estimator = Ridge(random_state = 42),
    cv = cv
)

ensemble_res('Stacking Ridge Meta-model', stacking_ridge)

[Stacking Ridge Meta-model] RMSLE: 0.1099 (+/- 0.0087)


In [8]:
# Bagging regressor
from sklearn.ensemble import BaggingRegressor

bagging_elasticnet = create_model(
    BaggingRegressor(
        estimator=ElasticNet(
            alpha=0.01,
            l1_ratio=0.5,
            max_iter=20000
        ),
        n_estimators=50,
        random_state=42,
        n_jobs=1
    ),
    preprocessor,
    scale=True
)

ensemble_res('Bagging ElasticNet', bagging_elasticnet)

[Bagging ElasticNet] RMSLE: 0.1110 (+/- 0.0073)


In [10]:
# Bagging regressor
from sklearn.ensemble import BaggingRegressor

bagging_ridge = create_model(
    BaggingRegressor(
        estimator=Ridge(
            alpha=100,
            max_iter=20000
        ),
        n_estimators=50,
        random_state=42,
        n_jobs=1
    ),
    preprocessor,
    scale=True
)

ensemble_res('Bagging Ridge', bagging_ridge)

[Bagging Ridge] RMSLE: 0.1176 (+/- 0.0090)


In [13]:
#AdaBoost and Stohastic Gradient Boost
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor

adaboost = create_model(
    AdaBoostRegressor(
        n_estimators=100,
        learning_rate=0.05,
        random_state=42
    ),
    preprocessor,
    scale=False
)
ensemble_res('AdaBoost', adaboost)

sgb = create_model(
    GradientBoostingRegressor(
        n_estimators = 100,
        subsample = 0.8,
        max_features = 'sqrt',
        random_state = 42
    ),
    preprocessor,
    scale = False
)
ensemble_res('Stohastic Gradient Boost', sgb)

[AdaBoost] RMSLE: 0.1872 (+/- 0.0076)
[Stohastic Gradient Boost] RMSLE: 0.1254 (+/- 0.0082)


In [17]:
results_ensembles = pd.DataFrame(ensemble_results).sort_values(by = 'RMSLE', ascending = True)
results_ensembles

,Ensemble,RMSLE,Std
0,Voting Regressor,0.108857,0.008013
2,Stacking Ridge Meta-model,0.109887,0.008702
3,Bagging ElasticNet,0.111043,0.007326
1,Weighted averaging Top Boosters,0.112830,0.008543
4,Bagging Ridge,0.117586,0.008985
7,Stohastic Gradient Boost,0.125441,0.008164
5,AdaBoost,0.187163,0.007563
6,AdaBoost,0.187163,0.007563
